# Eagle Eye YOLO training (generated from MODEL_TRAINING.md)
Template only: edit marked variables. Do not paste credentials; use Colab Secrets.

## Cell 1 — check GPU

In [ ]:
!nvidia-smi

## Cell 2 — install and print versions

In [ ]:
!pip install -q ultralytics
import sys, torch, ultralytics
print("python", sys.version.split()[0], "| torch", torch.__version__, "| ultralytics", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

## Cell 3 — mount Drive (optional, protects against disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/eagle-eye-training'   # EDIT
import os; os.makedirs(DRIVE_OUT, exist_ok=True)

## Cell 4 — get the dataset (template: choose ONE option and edit)

In [ ]:
# Option A: unzip a dataset zip stored on Drive (the zip must contain data.yaml, train/, valid/ ... directly, not inside an extra folder)
!mkdir -p /content/dataset
!unzip -q "/content/drive/MyDrive/cabinet_dataset.zip" -d /content/dataset   # EDIT path

# Option B: Roboflow export (public data). Store the key in Colab Secrets as ROBOFLOW_API_KEY.
# !pip install -q roboflow
# from google.colab import userdata
# from roboflow import Roboflow
# rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
# project = rf.workspace("WORKSPACE").project("PROJECT")      # EDIT from the dataset page
# dataset = project.version(1).download("yolov8")             # pick an Ultralytics-compatible format

## Cell 5 — build a clean data YAML and validate labels

In [ ]:
import yaml, os
DATA_ROOT = '/content/dataset'                 # EDIT: folder that holds data.yaml and train/ valid/ (Roboflow) or images/ labels/
orig = yaml.safe_load(open(f'{DATA_ROOT}/data.yaml'))
raw = orig['names']
names = dict(enumerate(raw)) if isinstance(raw, list) else {int(k): v for k, v in raw.items()}
NC = len(names)
print("classes:", NC, names)

if os.path.isdir(f'{DATA_ROOT}/images/train'):          # Ultralytics-style layout
    cfg = {'path': DATA_ROOT, 'train': 'images/train', 'val': 'images/val', 'names': names}
    if os.path.isdir(f'{DATA_ROOT}/images/test'): cfg['test'] = 'images/test'
else:                                                    # Roboflow export layout
    val_dir = 'valid' if os.path.isdir(f'{DATA_ROOT}/valid/images') else 'val'
    cfg = {'path': DATA_ROOT, 'train': 'train/images', 'val': f'{val_dir}/images', 'names': names}
    if os.path.isdir(f'{DATA_ROOT}/test/images'): cfg['test'] = 'test/images'
DATA_YAML = f'{DATA_ROOT}/eagle_eye_data.yaml'
yaml.safe_dump(cfg, open(DATA_YAML, 'w'))
print(open(DATA_YAML).read())
HAS_TEST = 'test' in cfg
# Upload scripts/dataset/validate_yolo_dataset.py to /content (or clone your repo), then:
!python /content/validate_yolo_dataset.py --root {DATA_ROOT} --nc {NC}

## Cell 6 — smoke test (1 epoch, tiny): proves the pipeline works

In [ ]:
from ultralytics import YOLO
MODEL = 'yolo11n.pt'      # pretrained baseline; yolo26n.pt is an option to compare later
m = YOLO(MODEL)
m.train(data=DATA_YAML, epochs=1, imgsz=320, batch=8, fraction=0.1, project='/content/runs', name='smoke', exist_ok=True)

## Cell 7 — real training (TEMPLATE: adjust to your dataset)

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=100,          # upper bound; early stopping decides
    patience=20,         # stop if val metrics do not improve for 20 epochs
    imgsz=640,           # reduce to 512/416 if GPU memory runs out
    batch=16,            # reduce to 8/4 on CUDA out-of-memory; -1 requests auto batch
    seed=0,
    project=DRIVE_OUT,   # runs saved to Drive so a disconnect does not lose them
    name='eagle_eye_v1',
    exist_ok=True,
)

## Cell 8 — evaluate on validation and test (test only once, at the end)

In [ ]:
best_path = f'{DRIVE_OUT}/eagle_eye_v1/weights/best.pt'
best = YOLO(best_path)
for split in (['val', 'test'] if HAS_TEST else ['val']):
    m_ = best.val(data=DATA_YAML, split=split, project=DRIVE_OUT, name=f'eval_{split}', exist_ok=True)
    print(split, 'precision', m_.box.mp, 'recall', m_.box.mr, 'mAP50', m_.box.map50, 'mAP50-95', m_.box.map)
    print('per-class mAP50-95:', dict(zip(m_.names.values(), m_.box.maps)))

## Cell 9 — qualitative check on sample images

In [ ]:
import glob
IMG_GLOB = f'{DATA_ROOT}/test/images/*' if os.path.isdir(f'{DATA_ROOT}/test/images') else f'{DATA_ROOT}/valid/images/*'
samples = glob.glob(IMG_GLOB)[:12]   # EDIT if you use the Ultralytics-style layout (images/test)
pred = best.predict(source=samples, conf=0.25, save=True, project=DRIVE_OUT, name='samples', exist_ok=True)

## Cell 10 — write the model manifest and download the checkpoint

In [ ]:
import hashlib, json, sys, datetime, torch, ultralytics
sha = hashlib.sha256(open(best_path, 'rb').read()).hexdigest()
manifest = {
  "model_family": "ultralytics-yolo",
  "base_checkpoint": MODEL,
  "checkpoint_path": "models/custom/best.pt",
  "checkpoint_sha256": sha,
  "class_names": {str(k): v for k, v in dict(names).items()} if isinstance(names, dict) else {str(i): n for i, n in enumerate(names)},
  "dataset": {"name": "FILL", "source_url": "FILL", "version": "FILL", "license": "FILL"},
  "training": {"epochs_max": 100, "patience": 20, "imgsz": 640, "batch": 16, "seed": 0, "date": datetime.date.today().isoformat()},
  "software": {"ultralytics": ultralytics.__version__, "torch": torch.__version__, "python": sys.version.split()[0]},
  "evaluation": {"split": "test", "precision": None, "recall": None, "map50": None, "map50_95": None, "status": "FILL FROM CELL 8"},
  "known_limitations": ["FILL"]
}
json.dump(manifest, open('/content/model_manifest.json', 'w'), indent=2)
from google.colab import files
files.download(best_path)
files.download('/content/model_manifest.json')